# Legacy sampler correction benchmark

Fairly compares the release script's source/class **window** balancing with source/class **participant** balancing on the unchanged Felius/Voisard/Sint data. No NONAN, synthetic, frozen NONAN, or RevalExo data are read.

In [1]:
from pathlib import Path
import sys,json,gc
import numpy as np,pandas as pd,torch
from torch.utils.data import DataLoader,TensorDataset,WeightedRandomSampler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score,balanced_accuracy_score,brier_score_loss
ROOT=Path.cwd().resolve(); ROOT=ROOT.parent if ROOT.name.lower()=='notebooks' else ROOT
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from models.stroke_gait_inception import StrokeGaitInception
P=ROOT/'data'/'processed'; D=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device:',D)
x=np.concatenate([np.load(P/'validated_acceleration_magnitude_windows_float32.npy'),np.load(P/'sint_maartenskliniek_external_windows_float32.npy')]); m=pd.concat([pd.read_csv(P/'validated_window_metadata.csv'),pd.read_csv(P/'sint_maartenskliniek_external_window_metadata.csv')],ignore_index=True); m=m[m.label.isin(['healthy','stroke'])].reset_index(drop=True); m['y']=m.label.eq('stroke').astype(int); m['source']=m.dataset_id; m['group']=m.participant_key.astype(str)
people=m[['group','source','y']].drop_duplicates().reset_index(drop=True); people['stratum']=people.source+'|'+people.y.astype(str)
def sampler_weights(frame,mode):
    cells=frame.groupby(['source','y']).size(); den=np.asarray(pd.MultiIndex.from_frame(frame[['source','y']]).map(cells),float); w=1/den
    if mode=='participant_source_class':
        pcounts=frame.groupby('group').size(); pden=np.asarray(pd.MultiIndex.from_frame(frame[['source','y']]).map(frame[['source','y','group']].drop_duplicates().groupby(['source','y']).size()),float); w=frame.group.map(1/pcounts).to_numpy()/pden
    return torch.tensor(w,dtype=torch.double)
def collect(net,arr,meta,mean,std,repeat,fold,mode,scope):
    with torch.inference_mode(): p=torch.sigmoid(net(torch.from_numpy(((arr-mean)/std).transpose(0,2,1).astype('float32')).to(D))).cpu().numpy()
    g=meta.assign(p=p).groupby(['group','source','y'],as_index=False).p.mean(); rows=[]
    for name,q in [('pooled',g),*[(s,g[g.source.eq(s)]) for s in sorted(g.source.unique())]]:
        if q.y.nunique()!=2: continue
        rows.append({'repeat':repeat,'fold':fold,'mode':mode,'scope':scope,'evaluation_source':name,'participants':len(q),'healthy':int((q.y==0).sum()),'stroke':int((q.y==1).sum()),'auroc':roc_auc_score(q.y,q.p),'balanced_accuracy':balanced_accuracy_score(q.y,q.p>=.5),'healthy_specificity':float((q.loc[q.y==0,'p']<.5).mean()),'brier':brier_score_loss(q.y,q.p)})
    return rows
rows=[]
for repeat in [42,137,202,404,909]:
    folds=StratifiedKFold(3,shuffle=True,random_state=repeat)
    for fold,(trp,vap) in enumerate(folds.split(people,people.stratum)):
        tg,vg=set(people.iloc[trp].group),set(people.iloc[vap].group); tr=m.group.isin(tg).to_numpy(); va=m.group.isin(vg).to_numpy()
        for mode in ['window_source_class','participant_source_class']:
            torch.manual_seed(repeat*100+fold); tx=x[tr]; mean,std=tx.reshape(-1,3).mean(0),tx.reshape(-1,3).std(0).clip(1e-4); z=torch.from_numpy(((tx-mean)/std).transpose(0,2,1).astype('float32')); y=torch.from_numpy(m.loc[tr,'y'].to_numpy('float32')); dl=DataLoader(TensorDataset(z,y),128,sampler=WeightedRandomSampler(sampler_weights(m.loc[tr],mode),len(z),replacement=True,generator=torch.Generator().manual_seed(repeat*1000+fold)))
            net=StrokeGaitInception().to(D); opt=torch.optim.AdamW(net.parameters(),1e-3,weight_decay=1e-4)
            for _ in range(8):
                net.train()
                for a,b in dl: opt.zero_grad(); loss=torch.nn.functional.binary_cross_entropy_with_logits(net(a.to(D)),b.to(D)); loss.backward(); opt.step()
            net.eval(); rows.extend(collect(net,x[va],m.loc[va],mean,std,repeat,fold,mode,'heldout_original')); del net,opt,dl,z,y; gc.collect(); torch.cuda.empty_cache() if D.type=='cuda' else None; print('complete',repeat,fold,mode)
out=pd.DataFrame(rows); out.to_csv(P/'legacy_sampler_correction_benchmark.csv',index=False); pooled=out[out.evaluation_source.eq('pooled')]; wide=pooled.pivot(index=['repeat','fold'],columns='mode',values=['auroc','balanced_accuracy','healthy_specificity','brier']); rng=np.random.default_rng(20260902); summary={}
for metric in ['auroc','balanced_accuracy','healthy_specificity','brier']:
    d=(wide[metric]['participant_source_class']-wide[metric]['window_source_class']).to_numpy(); bs=np.array([rng.choice(d,len(d),replace=True).mean() for _ in range(10000)]); summary[metric]={'mean_delta_participant_minus_window':float(d.mean()),'bootstrap_95_ci':[float(np.quantile(bs,.025)),float(np.quantile(bs,.975))],'nonnegative_units':int((d>=0).sum())}
print(json.dumps(summary,indent=2)); (P/'legacy_sampler_correction_benchmark_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')

device: cuda


complete 42 0 window_source_class


complete 42 0 participant_source_class


complete 42 1 window_source_class


complete 42 1 participant_source_class


...

complete 909 1 participant_source_class


complete 909 2 window_source_class


complete 909 2 participant_source_class


{
  "auroc": {
    "mean_delta_participant_minus_window": 0.0019812117087306827,
    "bootstrap_95_ci": [
      -0.004239248806634134,
      0.007805313052356526
    ],
    "nonnegative_units": 11
  },
  "balanced_accuracy": {
    "mean_delta_participant_minus_window": 0.018851780308607954,
    "bootstrap_95_ci": [
      -0.006178406414418136,
      0.04213954500045845
    ],
    "nonnegative_units": 11
  },
  "healthy_specificity": {
    "mean_delta_participant_minus_window": 0.06903996614716977,
    "bootstrap_95_ci": [
      0.0017303658086414903,
      0.13144320197354797
    ],
    "nonnegative_units": 13
  },
  "brier": {
    "mean_delta_participant_minus_window": -0.0005647278502475109,
    "bootstrap_95_ci": [
      -0.01380446918126022,
      0.012991958520940886
    ],
    "nonnegative_units": 7
  }
}


822

The participant-balanced sampler can replace the legacy release sampler only if its repeated pooled and worst-source behaviour is non-inferior, especially for healthy specificity and calibration. This benchmark establishes an internal development recipe; frozen external evaluation is a separate, one-time protocol after the recipe is locked.